In [57]:
from dotenv import load_dotenv

load_dotenv()

True

In [58]:
from langchain_core.tools import tool


@tool
def publish_article(slug: str) -> str:
    """Publish an article to the web."""
    return f"✅ Published {slug}"


@tool
def list_articles() -> str:
    """List all articles (read-only)."""
    return "article-1\narticle-2\narticle-3"


print("Tools defined")

Tools defined


In [59]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[publish_article, list_articles],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"publish_article": {"allowed_decisions": ["approve", "reject"]}},
        ),
    ],
)

print("Agent with HITL middleware created")

Agent with HITL middleware created


In [60]:
# Test 1: Read-only (no interrupt)
print("\n" + "=" * 60)
print("TEST 1: List articles (no approval needed)")
print("=" * 60)

config = {"configurable": {"thread_id": "1"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "List my articles"}]},
    config=config,
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")


TEST 1: List articles (no approval needed)

HumanMessage:
List my articles

AIMessage:
[{'id': 'toolu_014geDKRjaf3NfunFjHcZSno', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'list_articles', 'type': 'tool_use'}]

ToolMessage:
article-1
article-2
article-3

AIMessage:
You have 3 articles:

1. article-1
2. article-2
3. article-3

Would you like to publish any of these articles or perform any other actions with them?


In [61]:
# Test 2: Sensitive operation (interrupts)
print("\n" + "=" * 60)
print("TEST 2: Publish article (requires approval)")
print("=" * 60)

config = {"configurable": {"thread_id": "2"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Publish article-1"}]},
    config=config,
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")

if "__interrupt__" in result:
    desc = result["__interrupt__"][-1].value["action_requests"][-1]["description"]
    print(f"\n🔒 INTERRUPT: {desc}")


TEST 2: Publish article (requires approval)

HumanMessage:
Publish article-1

AIMessage:
[{'id': 'toolu_01GY4iHxcCYYoX4zGd2iB1VS', 'caller': {'type': 'direct'}, 'input': {'slug': 'article-1'}, 'name': 'publish_article', 'type': 'tool_use'}]

🔒 INTERRUPT: Tool execution requires approval

Tool: publish_article
Args: {'slug': 'article-1'}


In [62]:
from langgraph.types import Command

# Test 3: Approve and resume
print("\n" + "=" * 60)
print("TEST 3: Approve and resume")
print("=" * 60)

result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,  # Same thread
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")


TEST 3: Approve and resume

HumanMessage:
Publish article-1

AIMessage:
[{'id': 'toolu_01GY4iHxcCYYoX4zGd2iB1VS', 'caller': {'type': 'direct'}, 'input': {'slug': 'article-1'}, 'name': 'publish_article', 'type': 'tool_use'}]

ToolMessage:
✅ Published article-1

AIMessage:
Done! I've successfully published the article with slug "article-1".


In [63]:
# Test 4: Reject request
print("\n" + "=" * 60)
print("TEST 4: Reject publication")
print("=" * 60)

config = {"configurable": {"thread_id": "3"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Publish article-2"}]},
    config=config,
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")

if "__interrupt__" in result:
    print("\n🔒 Approval required")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "message": "Not ready yet"}]}),
        config=config,
    )

    for msg in result["messages"]:
        print(f"\n{msg.__class__.__name__}:")
        print(f"{msg.content}")


TEST 4: Reject publication

HumanMessage:
Publish article-2

AIMessage:
[{'id': 'toolu_015daRh6PqBkBiEF65G8JTYT', 'caller': {'type': 'direct'}, 'input': {'slug': 'article-2'}, 'name': 'publish_article', 'type': 'tool_use'}]

🔒 Approval required

HumanMessage:
Publish article-2

AIMessage:
[{'id': 'toolu_015daRh6PqBkBiEF65G8JTYT', 'caller': {'type': 'direct'}, 'input': {'slug': 'article-2'}, 'name': 'publish_article', 'type': 'tool_use'}]

ToolMessage:
Not ready yet

AIMessage:
It appears that article-2 is not ready to be published yet. This could mean:
- The article is still in draft status
- It hasn't been fully prepared or reviewed
- There may be missing content or metadata

Would you like me to list all available articles to check their status?
